# Pilot study prototype

Thank you for participating.

This prototype contains short interactive programming exercises. The aim is to test the design of the exercises, not to test you as a programmer.

Please think aloud while solving the tasks. Say what you notice, what you are unsure about, and why you choose a particular answer.

You may use the hint, solution, and walkthrough buttons when they become available. They are part of the prototype, so using them is not a failure.

There are four tasks in total:
1. two tasks about execution order and sequential processes;
2. two tasks about program structure, decomposition, and abstraction.


In [1]:

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import re


def _show(widget):
    widget.layout.display = ''


def normalize_list_answer(text):
    return [part.strip() for part in text.split(',') if part.strip()]


def create_order_task(task_id, correct_order, hint_md, solution_md, walkthrough_md, n_lines):
    answer_input = widgets.Text(
        value='',
        placeholder='e.g. 2,1,3,4',
        description='Order:',
        layout=widgets.Layout(width='360px')
    )

    check_button = widgets.Button(description='Check answer', button_style='success')
    hint_button = widgets.Button(description='Show hint', layout=widgets.Layout(display='none'))
    walkthrough_button = widgets.Button(description='Show walkthrough', layout=widgets.Layout(display='none'))
    solution_button = widgets.Button(description='Show solution', layout=widgets.Layout(display='none'))

    feedback = widgets.Output()
    support = widgets.Output()
    wrong_attempts = {'count': 0}

    def on_check_clicked(_):
        with feedback:
            clear_output()
            user_answer = normalize_list_answer(answer_input.value)

            if len(user_answer) != n_lines:
                print(f'Please enter exactly {n_lines} line numbers, separated by commas.')
                return

            if sorted(user_answer) != sorted(correct_order):
                print('Please use each line number exactly once.')
                return

            if user_answer == correct_order:
                display(Markdown('**Correct.**'))
            else:
                wrong_attempts['count'] += 1
                if wrong_attempts['count'] == 1:
                    _show(hint_button)
                    display(Markdown('**Not quite.** You can now choose to show a hint.'))
                else:
                    _show(hint_button)
                    _show(walkthrough_button)
                    _show(solution_button)
                    display(Markdown('**Still not quite.** You can now choose to see the walkthrough or the solution.'))

    def show_hint(_):
        with support:
            clear_output()
            display(Markdown(hint_md))

    def show_solution(_):
        with support:
            clear_output()
            display(Markdown(solution_md))

    def show_walkthrough(_):
        with support:
            clear_output()
            display(Markdown(walkthrough_md))

    check_button.on_click(on_check_clicked)
    hint_button.on_click(show_hint)
    solution_button.on_click(show_solution)
    walkthrough_button.on_click(show_walkthrough)

    display(widgets.VBox([
        answer_input,
        widgets.HBox([check_button, hint_button, walkthrough_button, solution_button]),
        feedback,
        support
    ]))


def normalize_blocks(text):
    groups = text.split(';')
    normalized = []
    for group in groups:
        items = [part.strip() for part in group.split(',') if part.strip()]
        if items:
            normalized.append(items)
    return normalized


def same_block_membership(user_blocks, correct_blocks):
    """Compare groups by membership, not by the order of line numbers inside each group."""
    return [set(block) for block in user_blocks] == [set(block) for block in correct_blocks]


def create_block_task(correct_blocks, hint_md, solution_md, walkthrough_md, all_lines):
    answer_input = widgets.Text(
        value='',
        placeholder='e.g. 1,2; 3,4; 5',
        description='Blocks:',
        layout=widgets.Layout(width='460px')
    )

    check_button = widgets.Button(description='Check answer', button_style='success')
    hint_button = widgets.Button(description='Show hint', layout=widgets.Layout(display='none'))
    walkthrough_button = widgets.Button(description='Show walkthrough', layout=widgets.Layout(display='none'))
    solution_button = widgets.Button(description='Show solution', layout=widgets.Layout(display='none'))

    feedback = widgets.Output()
    support = widgets.Output()
    wrong_attempts = {'count': 0}

    def on_check_clicked(_):
        with feedback:
            clear_output()
            user_blocks = normalize_blocks(answer_input.value)

            if len(user_blocks) != len(correct_blocks):
                print(f'Please enter exactly {len(correct_blocks)} groups separated by semicolons.')
                return

            flat = [item for group in user_blocks for item in group]
            if sorted(flat) != sorted(all_lines) or len(flat) != len(set(flat)):
                print('Please use each line number exactly once.')
                return

            if same_block_membership(user_blocks, correct_blocks):
                display(Markdown('**Correct.**'))
            else:
                wrong_attempts['count'] += 1
                if wrong_attempts['count'] == 1:
                    _show(hint_button)
                    display(Markdown('**Not quite.** You can now choose to show a hint.'))
                else:
                    _show(hint_button)
                    _show(walkthrough_button)
                    _show(solution_button)
                    display(Markdown('**Still not quite.** You can now choose to see the walkthrough or the solution.'))

    def show_hint(_):
        with support:
            clear_output()
            display(Markdown(hint_md))

    def show_solution(_):
        with support:
            clear_output()
            display(Markdown(solution_md))

    def show_walkthrough(_):
        with support:
            clear_output()
            display(Markdown(walkthrough_md))

    check_button.on_click(on_check_clicked)
    hint_button.on_click(show_hint)
    solution_button.on_click(show_solution)
    walkthrough_button.on_click(show_walkthrough)

    display(widgets.VBox([
        answer_input,
        widgets.HBox([check_button, hint_button, walkthrough_button, solution_button]),
        feedback,
        support
    ]))


def parse_line_groups(text):
    # Accept formats like "1,2 and 4,5", "1,2; 4,5", or "Repeated lines: 1,2 and 4,5".
    text = text.lower().replace('&', 'and').replace(';', 'and')
    raw_groups = [part.strip() for part in text.split('and') if part.strip()]
    groups = []

    for group in raw_groups:
        numbers = re.findall(r'\d+', group)
        if len(numbers) >= 2:
            groups.append(tuple(sorted(numbers)))
    return sorted(groups)


def normalize_expected_group_alternatives(expected_group_alternatives):
    normalized = []
    for alternative in expected_group_alternatives:
        normalized.append(sorted([tuple(sorted(group)) for group in alternative]))
    return normalized



def parse_inputs(text):
    # Accept formats like "name, price", "Inputs: title and price", or "product name + price".
    return [token.lower() for token in re.findall(r'[A-Za-z_]\w*', text)]


def input_answer_is_reasonable(tokens):
    # Keep this deliberately flexible for the pilot. The explanation matters more than exact wording.
    has_price = any('price' in token for token in tokens)
    has_name_like_input = any(
        keyword in token
        for token in tokens
        for keyword in ['name', 'title', 'product', 'item']
    )
    return has_price and has_name_like_input

def create_abstraction_task(expected_group_alternatives, hint_md, solution_md, walkthrough_md):
    lines_input = widgets.Text(
        value='',
        placeholder='e.g. 1,2 and 3,4',
        description='Repeated:',
        layout=widgets.Layout(width='520px')
    )

    name_input = widgets.Text(
        value='',
        placeholder='e.g. example_function_name',
        description='Function name:',
        layout=widgets.Layout(width='520px')
    )

    inputs_input = widgets.Text(
        value='',
        placeholder='e.g. example_arg1, example_arg2',
        description='Inputs:',
        layout=widgets.Layout(width='520px')
    )

    check_button = widgets.Button(description='Check answer', button_style='success')
    hint_button = widgets.Button(description='Show hint', layout=widgets.Layout(display='none'))
    walkthrough_button = widgets.Button(description='Show walkthrough', layout=widgets.Layout(display='none'))
    solution_button = widgets.Button(description='Show solution', layout=widgets.Layout(display='none'))

    feedback = widgets.Output()
    support = widgets.Output()
    wrong_attempts = {'count': 0}

    expected_alternatives = normalize_expected_group_alternatives(expected_group_alternatives)

    def acceptable_name(name):
        name = name.strip()
        if not name:
            return False
        if ' ' in name or '-' in name:
            return False
        if len(name) < 3:
            return False
        # Keep this deliberately permissive for the pilot. The explanation matters more than an exact name.
        return bool(re.match(r'^[A-Za-z_]\w*$', name))

    def on_check_clicked(_):
        with feedback:
            clear_output()
            groups = parse_line_groups(lines_input.value)
            name = name_input.value.strip()
            input_tokens = parse_inputs(inputs_input.value)

            if not lines_input.value.strip() or not name or not inputs_input.value.strip():
                print('Please answer all three parts.')
                return

            line_check = groups in expected_alternatives
            name_check = acceptable_name(name)
            inputs_check = input_answer_is_reasonable(input_tokens)

            if line_check and name_check and inputs_check:
                display(Markdown('**Reasonable answer.** You identified repeated structure and suggested a plausible abstraction. The facilitator may ask you to explain why you chose those lines and inputs.'))
            else:
                wrong_attempts['count'] += 1
                if wrong_attempts['count'] == 1:
                    _show(hint_button)
                    display(Markdown('**Not quite.** You can now choose to show a hint.'))
                else:
                    _show(hint_button)
                    _show(walkthrough_button)
                    _show(solution_button)
                    display(Markdown('**Still not quite.** You can now choose to see the walkthrough or the solution.'))

    def show_hint(_):
        with support:
            clear_output()
            display(Markdown(hint_md))

    def show_solution(_):
        with support:
            clear_output()
            display(Markdown(solution_md))

    def show_walkthrough(_):
        with support:
            clear_output()
            display(Markdown(walkthrough_md))

    check_button.on_click(on_check_clicked)
    hint_button.on_click(show_hint)
    solution_button.on_click(show_solution)
    walkthrough_button.on_click(show_walkthrough)

    display(widgets.VBox([
        lines_input,
        name_input,
        inputs_input,
        widgets.HBox([check_button, hint_button, walkthrough_button, solution_button]),
        feedback,
        support
    ]))


# Concept 1: Sequential Processes

In these tasks, the code lines are shown in the wrong order.

Your task is to decide the correct execution order.

Write your answer as a comma-separated sequence of line numbers, for example:

`2,1,3,4`


## Task 1: Variable dependency

Arrange the lines so each variable is defined before it is used.

1. `discount = price * 0.10`  
2. `label = "Final price: " + str(final_price)`  
3. `price = base_price + fee`  
4. `final_price = price - discount`  
5. `fee = base_price * 0.05`  
6. `base_price = 100`


In [2]:

create_order_task(
    task_id='vd',
    correct_order=['6', '5', '3', '1', '4', '2'],
    n_lines=6,
    hint_md="""**Hint:** Start with the only line that can run without using a value created by another line. Then follow the chain of dependencies.""",
    solution_md="""**Solution:** `6,5,3,1,4,2`""",
    walkthrough_md="""**Walkthrough:**

1. `base_price = 100` must come first, because it creates `base_price`.
2. `fee = base_price * 0.05` can now run, because `base_price` exists.
3. `price = base_price + fee` can now run, because both `base_price` and `fee` exist.
4. `discount = price * 0.10` can now run, because `price` exists.
5. `final_price = price - discount` can now run, because both `price` and `discount` exist.
6. `label = "Final price: " + str(final_price)` comes last, because it depends on `final_price`.

The important idea is that execution order is constrained by variable dependencies, even when the code is not written as a simple `x`, `y`, `z` chain.""",
)


## Task 2: Loop setup

Arrange the lines so the program correctly computes a total.

1. `total = total + n`  
2. `numbers = [1, 2, 3]`  
3. `total = len(numbers) * 0`  
4. `for n in numbers:`


In [3]:

create_order_task(
    task_id='loop',
    correct_order=['2', '3', '4', '1'],
    n_lines=4,
    hint_md="""**Hint:** The list must exist before it can be used to initialize the accumulator or run the loop.""",
    solution_md="""**Solution:** `2,3,4,1`""",
    walkthrough_md="""**Walkthrough:**

1. `numbers = [1, 2, 3]` creates the data the loop will use.
2. `total = len(numbers) * 0` initializes the accumulator. This line uses `numbers`, so it must come after the list has been created.
3. `for n in numbers:` starts the repeated process.
4. `total = total + n` is the loop body, which updates the accumulator once for each number.

The important idea is that setup must happen before the loop, and the loop body is written once but executed repeatedly.""",
)


# Concept 2: Abstraction and Decomposition

In these tasks, the focus is not only what the program does line by line, but how the program is structured.

You will group lines into meaningful parts and identify repeated logic that could be abstracted into a function.


## Task 3: Group lines into meaningful blocks

Below is a small Python program.

Divide the lines into **3 logical blocks**:

1. setup  
2. processing  
3. output  

Write your answer as three groups of line numbers separated by semicolons.

The order of the three blocks matters, but the order of line numbers inside each block is not important.

Example format:

`1,2; 3,4; 5`

### Code

1. `prices = [20, 35, 15]`  
2. `discount = 0.8`  
3. `discounted_prices = []`  
4. `for price in prices:`  
5. `    discounted_prices.append(price * discount)`  
6. `total = sum(discounted_prices)`  
7. `print(total)`


In [4]:

create_block_task(
    correct_blocks=[['1', '2', '3'], ['4', '5', '6'], ['7']],
    all_lines=['1', '2', '3', '4', '5', '6', '7'],
    hint_md="""**Hint:** Think about what information is prepared first, where the main calculation happens, and where the result is shown.""",
    solution_md="""**Solution:** `1,2,3; 4,5,6; 7`\n\nThe exact order inside each group is less important here than whether the lines belong to the same meaningful part of the program.""",
    walkthrough_md="""**Walkthrough:**

- Lines 1-3 are setup: they create the input data, the discount value, and an empty list for results.
- Lines 4-6 are processing: they apply the discount and compute the total.
- Line 7 is output: it displays the final result.

The important idea is that a program can be understood as meaningful parts, not only as individual lines.""",
)


## Task 4: Repeated structure and abstraction

Below is a small Python program.

The code creates two price labels. The two parts are not identical, but they have a similar structure and purpose.

Your task has three parts:

1. Identify which lines contain the repeated label-building logic.
2. Suggest a short function name that could abstract that repeated work.
3. Write what inputs the function would need.

Write your answer in this format:

`Repeated lines: 1,2 and 5,6`  
`Function name: example_function_name`  
`Inputs: example_arg1, example_arg2`

### Code

1. `book_title = "Python 101"`  
2. `book_price = 120`  
3. `book_label = book_title + ": " + str(book_price) + " kr."`  
4. `print(book_label)`  
5. `lamp_name = "Desk lamp"`  
6. `lamp_price = 80`  
7. `lamp_label = lamp_name + ": " + str(lamp_price) + " kr."`  
8. `print(lamp_label)`



In [5]:

create_abstraction_task(
    expected_group_alternatives=[
        [('1', '2', '3'), ('5', '6', '7')],
        [('1', '2', '3', '4'), ('5', '6', '7', '8')],
    ],
    hint_md="""**Hint:** Look for two groups that do the same kind of work, even though the variable names are different. Ask yourself which values change between the two groups, and which structure stays the same.""",
    solution_md="""**One strong answer:**

Repeated lines: `1,2,3 and 5,6,7`  
Function name: `make_price_label`  
Inputs: `name, price`

This treats the abstraction as a function that builds a price label. The `print` lines stay outside the function because printing the label is a separate responsibility from creating it.

It is also possible to include lines `4` and `8` if you think the function should both create and display the label. In that case, a name such as `print_price_label` or `show_price_label` would be more precise.""",
    walkthrough_md="""**Walkthrough:**

The two blocks are not identical, but they have the same underlying purpose.

Lines 1-3 create a label for a book:

1. define a product title;
2. define a price;
3. combine the title and price into a text label.

Lines 5-7 do the same kind of work for a lamp:

1. define a product name;
2. define a price;
3. combine the name and price into a text label.

A useful abstraction would separate the general label-building idea from the specific products.

For example:

```python
def make_price_label(name, price):
    return name + ": " + str(price) + " kr."
```

Then the two products can reuse the same idea with different inputs:

```python
book_label = make_price_label("Python 101", 120)
lamp_label = make_price_label("Desk lamp", 80)
```

The important point is not the exact function name, but noticing that `book_title` and `lamp_name` play the same role, and that `book_price` and `lamp_price` play the same role."""
)


## End

Thank you. The facilitator will now ask a few follow-up questions.